In [1]:
import os
import random
from pathlib import Path

# ==========================================
# BƯỚC 1: MANG CODE VÀO NHÀ (TẢI LẠI TỪ GITHUB)
# ==========================================
%cd /kaggle/working/
# Nếu thư mục đã tồn tại thì xóa đi tải lại cho sạch
!rm -rf /kaggle/working/Secure-Virtual-Assistant-with-Speaker-Recognition
!git clone https://github.com/23120340/Secure-Virtual-Assistant-with-Speaker-Recognition.git

# Đi vào trong nhà và cài đặt đồ nghề
%cd /kaggle/working/Secure-Virtual-Assistant-with-Speaker-Recognition
!pip install -r training/requirements.txt


# ==========================================
# BƯỚC 2: TẠO LẠI CUỐN "MỤC LỤC" DỮ LIỆU
# ==========================================
VIVOS_WAVES_ROOT = Path("/kaggle/input/datasets/kynthesis/vivos-vietnamese-speech-corpus-for-asr/vivos/train/waves")
OUTPUT_SPLIT_FILE = Path("/kaggle/working/vivos_iden_split.txt")

lines = []
for spk_dir in VIVOS_WAVES_ROOT.iterdir():
    if spk_dir.is_dir():
        for wav_file in spk_dir.glob("*.wav"):
            rel_path = wav_file.relative_to(VIVOS_WAVES_ROOT)
            formatted_path = str(rel_path).replace("\\", "/")
            r = random.random()
            if r < 0.8: split_id = 1
            elif r < 0.9: split_id = 2
            else: split_id = 3
            lines.append(f"{split_id} {formatted_path}\n")

with open(OUTPUT_SPLIT_FILE, "w", encoding="utf-8") as f:
    f.writelines(lines)
print(f"\n[+] Đã tạo lại mục lục thành công với {len(lines)} file!")

# ==========================================
# BƯỚC 2.5: TẠO "ĐỀ THI" XÁC THỰC (TRIAL PAIRS) CHO SV
# ==========================================
import itertools

TRIAL_FILE = Path("/kaggle/working/vivos_veri_test.txt")
spk2files = {}

# Gom nhóm các file lại theo từng người nói (Speaker)
with open(OUTPUT_SPLIT_FILE, "r") as f:
    for line in f:
        split_id, path = line.strip().split()
        if split_id == "3": # Chỉ lấy tập Test (Đề thi) để tạo cặp
            spk = path.split("/")[0]
            spk2files.setdefault(spk, []).append(path)

pos_pairs, neg_pairs = [], []
speakers = list(spk2files.keys())

# Tạo cặp Positive (Cùng người)
for spk, files in spk2files.items():
    if len(files) >= 2:
        # Lấy tối đa 5 cặp cho mỗi người để tránh file quá lớn
        for p1, p2 in itertools.combinations(files[:5], 2):
            pos_pairs.append((1, p1, p2))

# Tạo cặp Negative (Khác người)
for i in range(len(speakers)):
    for j in range(i + 1, len(speakers)):
        if spk2files[speakers[i]] and spk2files[speakers[j]]:
            f1 = random.choice(spk2files[speakers[i]])
            f2 = random.choice(spk2files[speakers[j]])
            neg_pairs.append((0, f1, f2))

# Cân bằng số lượng câu hỏi Positive và Negative
k = min(len(pos_pairs), len(neg_pairs))
trials = random.sample(pos_pairs, k) + random.sample(neg_pairs, k)
random.shuffle(trials)

with open(TRIAL_FILE, "w") as f:
    for label, p1, p2 in trials:
        f.write(f"{label} {p1} {p2}\n")
print(f"[+] Đã tạo danh sách đề thi xác thực với {len(trials)} cặp câu hỏi!")
# ==========================================
# BƯỚC 3: BẤM NÚT CHO AI ĐI HỌC (TRAINING)
# ==========================================
!python training/train_ecapa.py \
    --data_root "/kaggle/input/datasets/kynthesis/vivos-vietnamese-speech-corpus-for-asr/vivos/train/waves" \
    --split_file "/kaggle/working/vivos_iden_split.txt" \
    --save_dir "/kaggle/working/checkpoints_vivos" \
    --epochs 20 \
    --batch_size 64

/kaggle/working
Cloning into 'Secure-Virtual-Assistant-with-Speaker-Recognition'...
remote: Enumerating objects: 725, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 725 (delta 22), reused 78 (delta 20), pack-reused 625 (from 1)
Receiving objects: 100% (725/725), 76.95 MiB | 40.14 MiB/s, done.
Resolving deltas: 100% (414/414), done.
/kaggle/working/Secure-Virtual-Assistant-with-Speaker-Recognition
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 29.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 50.6 MB/s eta 0:00:00
  Attempting uninstall: ruamel.yaml
    Found existing installation: ruamel.yaml 0.19.1
    Uninstalling ruamel.yaml-0.19.1:
      Successfully uninstalled ruamel.yaml-0.19.1

[+] Đã tạo lại mục lục thành công với 11660 file!
[+] Đã tạo danh sách đề thi xác thực với 920 cặp câu hỏ